# <u>FEATURE ENGINEERING:</u>

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('../data/processed/clean_cars.csv')

In [7]:
df

,manufacturer,model,year,mileage,accidents_or_damage,one_owner,personal_use_only,seller_name,seller_rating,driver_rating,...,engine_liters,engine_cylinders,engine_type,transmission_simple,drivetrain_simple,fuel_simple,mpg_min,mpg_max,seller_rating_cat,driver_rating_cat
0,Acura,ILX Hybrid 1.5L,2013,47645.0,1,1,1,Kars Today,NaN,4.4,...,1.5,4,hybrid,automatic,Front-wheel Drive,Hybrid,38.0,39.0,sin valoración,4-5
1,Acura,ILX Hybrid 1.5L,2013,53422.0,0,1,1,Weiss Toyota of South County,4.3,4.4,...,1.5,4,hybrid,automatic,Front-wheel Drive,Hybrid,38.0,39.0,4-5,4-5
2,Acura,ILX Hybrid 1.5L,2013,117598.0,0,1,1,Apple Tree Acura,NaN,4.4,...,1.5,4,hybrid,automatic,Front-wheel Drive,Hybrid,38.0,39.0,sin valoración,4-5
3,Acura,ILX Hybrid 1.5L,2013,114865.0,1,0,1,Herb Connolly Chevrolet,3.7,4.4,...,1.5,4,hybrid,automatic,Front-wheel Drive,Hybrid,38.0,39.0,3-4,4-5
4,Acura,ILX Hybrid 1.5L,2013,62042.0,0,0,1,Kalidy Kia,2.2,4.4,...,1.5,4,hybrid,automatic,Front-wheel Drive,Hybrid,38.0,39.0,2-3,4-5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
591093,Volvo,S60 T5,2020,26781.0,1,1,1,Jenkins Volvo of Ocala,5.0,4.9,...,2.0,4,turbo,automatic,Front-wheel Drive,Gasoline,23.0,34.0,4-5,4-5
591094,Volvo,S60 B5 Momentum,2022,22877.0,0,1,0,Volvo Cars Danbury,4.2,4.2,...,2.0,4,turbo,automatic,All-wheel Drive,Gasoline,25.0,33.0,4-5,4-5
591095,Volvo,S60 T5,2014,92000.0,0,0,1,Dapper Car Sales,NaN,4.8,...,2.5,5,turbo,automatic,Front-wheel Drive,Gasoline,21.0,30.0,sin valoración,4-5
591096,Volvo,S60 T5 Platinum,2013,132000.0,1,0,0,Legend Auto Sales Inc.,4.6,4.7,...,2.5,5,turbo,automatic,All-wheel Drive,Gasoline,20.0,29.0,4-5,4-5


## <u>1. Nuevas variables numéricas:</u>

### <u>'car_age'</u>

##### Se crea car_age porque la antigüedad del coche es más interpretativa que el año y suele correlacionar con el precio.

In [8]:
data_year = 2024
df['car_age'] = data_year - df['year']

### <u>'mileage_per_year'</u>

#### Se crea esta variable para diferenciar entre coches con muchos km pero muy viejo vs un coche muy usado.

In [9]:
df['car_age_clip'] = df['car_age'].clip(lower=1)
df['mileage_per_year'] = df['mileage'] / df['car_age_clip']

#### Próximamente se decidirá si usar car_age y mileage_per_yer, y quizá prescindír de mileage o mentenerlo para comparar.

### <u>'mpg_mean'</u>

##### Se resume el rango de consumo en una única medida central y eliminamos ambas columnas de máximos y mínimos.

In [10]:
df['mpg_mean'] = (df['mpg_min'] + df['mpg_max']) / 2
df.drop(columns=['mpg_min', 'mpg_max'], inplace=True)

### <u>'engine_power_index'</u>

##### Hemos decidido combinar tamaño y cilindros en una variable más sencilla. Aunque no sea físico al 100%, pero como atributo puede tener sentido (motores más grandes y con más cilindros -> mayor potencia -> precio)

In [11]:
df['engine_power_index'] = df['engine_liters'] * df['engine_cylinders']

## <u>2.Transformaciones:</u>

##### Las variables como price, mileage y driver_reviews_num suelen estar muy sesgadas y por lo tanto:

### <u>'log_price'</u>

In [12]:
df['log_price'] = np.log(df['price'])

##### Luego podremos probar dos enfoques, uno con target = price y otro con target = log_price.

### <u>'log_mileage'</u>

In [13]:
df['log_mileage'] = np.log1p(df['mileage'])

### <u>'log_driver_reviews_num'</u>

In [14]:
df['log_driver_reviews_num'] = np.log1p(df['driver_reviews_num'])

## <u>3. Tratamiento de categorías grandes:</u>

##### Algunas variables como model y seller_name tienen una gran cantidad de categorías. Esto explota el número de columnas.

### <u>'seller_name'</u>

##### Seller_name se ha decidido eliminar ya que es una variable con cardinalidad extremadamente alta (14.190 valores distintos). Funciona casi como un identificador arbitrario.

In [15]:
df.drop(columns=['seller_name'], inplace=True)

### <u>'model'</u>

##### Para model, lo dejaremos tal cual en el modelo pero usando OneHotEncoder con min_frequency en el pipeline de scikit-learn para agrupar modelos muy raro automáticamente.

## <u>4. Selección de variables:</u>

Con todo lo anterior, podemos acabar definiendo las variables así:

**Variables numéricas:**

- `car_age`
- `mileage` o `log_mileage`
- `mileage_per_year`
- `engine_liters`
- `engine_cylinders`
- `engine_power_index`
- `mpg_mean`
- `driver_reviews_num` o `log_driver_reviews_num`
- `seller_rating` o `seller_rating_cat` (como categórica)
- `driver_rating` o `driver_rating_cat` (como categórica)

**Variables categóricas:**

- `manufacturer`
- `model`
- `fuel_simple`
- `transmission_simple`
- `drivetrain_simple`
- `accidents_or_damage`
- `one_owner`
- `personal_use_only`

Además, se han eliminado algunas variables originales porque:

- Han sido sustituidas por versiones refinadas (`mpg_min` / `mpg_max` frente a `mpg_mean`).
- Son difíciles de generalizar y presentan una cardinalidad muy alta  
  (como `seller_name` completa, por ejemplo).


## <u>5. GUARDÁMOS DATASET LISTO PARA EL MODELO:</u>

In [16]:
df.to_csv('../data/processed/model_cars.csv', index=False)